# Sistemas Distribuídos e Programação Paralela
## Respostas da Lista 06
### Matheus Sarno Muiños

# Parte 1 — De Lock para Condition

## Conceito

`Lock` responde a **quem pode entrar na região crítica**. Já `Condition` resolve outro problema: uma thread pode precisar esperar até que um estado compartilhado mude.

Exemplo: o consumidor não deve testar o tempo todo se o dado chegou. Ele chama `wait()`, libera o lock e aguarda uma notificação.

## Trecho importante

```python
with condition:
    while not dado_disponivel:
        condition.wait()
```

O `while` é essencial porque a thread deve confirmar a condição novamente quando acordar. `notify()` apenas acorda uma thread; não transfere o dado e não garante que ela continuará imediatamente.

## Respostas

1. O consumidor espera enquanto `dado_disponivel` for `False`.
2. O produtor deve chamar `notify()` depois de alterar o estado para disponível.
3. O consumidor não pode continuar antes de `dado_disponivel` se tornar `True`.
4. Usamos `while`, e não `if`, porque a condição precisa ser verificada novamente após a notificação.
5. `notify()` apenas acorda uma thread que espera; o dado continua nas variáveis compartilhadas.
6. O consumidor espera uma mudança de estado, e não diretamente a execução do produtor.

## Análise

`Condition` combina exclusão mútua e coordenação. Enquanto espera, a thread não fica consumindo processador em um laço contínuo e libera o lock para que o produtor possa trabalhar.

In [ ]:
import threading
import time

condition = threading.Condition()
dado_disponivel = False
dado = None


def consumidor():
    global dado_disponivel, dado
    with condition:
        while not dado_disponivel:
            print("Consumidor: dado indisponivel; aguardando...")
            condition.wait()
        print(f"Consumidor: recebi o dado {dado}")


def produtor():
    global dado_disponivel, dado
    time.sleep(1)
    with condition:
        dado = 42
        dado_disponivel = True
        print("Produtor: dado produzido.")
        condition.notify()


consumidor_thread = threading.Thread(target=consumidor)
produtor_thread = threading.Thread(target=produtor)
consumidor_thread.start()
produtor_thread.start()
consumidor_thread.join()
produtor_thread.join()

# Analise do resultado: o consumidor bloqueia até o produtor alterar
# dado_disponivel e chamar notify().

# Parte 2 — Produtor–Consumidor com buffer limitado

## Conceito e código central

O produtor espera quando `len(buffer) >= MAX_SIZE`. O consumidor espera quando `len(buffer) == 0`. Ambos usam a mesma `Condition` para alterar o buffer e notificar as outras threads.

```python
with condition:
    while len(buffer) >= MAX_SIZE:
        condition.wait()
    buffer.append(item)
    condition.notify_all()
```

## Respostas

1. Um produtor com buffer cheio chama `wait()` e libera o lock.
2. Um consumidor com buffer vazio chama `wait()` e libera o lock.
3. O lock precisa ser liberado para que a outra função possa alterar o buffer.
4. O produtor chama `notify_all()` depois de inserir um item.
5. O consumidor chama `notify_all()` depois de remover um item.
6. As sentinelas `None` só são colocadas depois que os produtores terminam, para não sinalizar encerramento antes de todos os itens reais serem produzidos.

## Análise

A sincronização evita acesso inconsistente ao buffer. A produção mais rápida faz o buffer encher e bloqueia produtores; o consumo mais rápido faz o buffer esvaziar e bloqueia consumidores. O `while` sempre revalida a condição depois de uma thread acordar.

In [ ]:
import random
import threading
import time

MAX_SIZE = 4
NUM_ITEMS = 30
buffer = []
condition = threading.Condition()
esperas_produtores = 0
esperas_consumidores = 0


def produtor(pid, quantidade):
    global esperas_produtores
    for numero in range(quantidade):
        item = f"P{pid}-{numero + 1}"
        with condition:
            while len(buffer) >= MAX_SIZE:
                esperas_produtores += 1
                print(f"[P{pid}] Buffer cheio. Esperando...")
                condition.wait()
            buffer.append(item)
            print(f"[P{pid}] Produziu {item} | Buffer: {list(buffer)}")
            condition.notify_all()
        time.sleep(random.uniform(0.01, 0.04))


def consumidor(cid):
    global esperas_consumidores
    while True:
        with condition:
            while not buffer:
                esperas_consumidores += 1
                print(f"[C{cid}] Buffer vazio. Esperando...")
                condition.wait()
            item = buffer.pop(0)
            condition.notify_all()
        if item is None:
            print(f"[C{cid}] Encerrando.")
            return
        print(f"[C{cid}] Consumiu {item} | Buffer: {list(buffer)}")
        time.sleep(random.uniform(0.01, 0.04))


produtores = [threading.Thread(target=produtor, args=(pid, NUM_ITEMS // 3)) for pid in range(3)]
consumidores = [threading.Thread(target=consumidor, args=(cid,)) for cid in range(2)]
for thread in consumidores + produtores:
    thread.start()
for thread in produtores:
    thread.join()
with condition:
    buffer.extend([None, None])
    condition.notify_all()
for thread in consumidores:
    thread.join()

print(f"Esperas dos produtores: {esperas_produtores}")
print(f"Esperas dos consumidores: {esperas_consumidores}")

# A quantidade de esperas varia com o escalonamento e com as pausas,
# mas todos os itens devem ser produzidos e consumidos corretamente.

# Parte 3 — Experimentos dirigidos com Condition

## Experimento 1 — Tamanho do buffer

Com `MAX_SIZE = 1`, o produtor tende a esperar mais porque o buffer fica cheio rapidamente. Com `MAX_SIZE = 10`, há mais espaço e os produtores bloqueiam menos. O resultado lógico permanece correto: todos os itens continuam sendo produzidos e consumidos.

## Experimento 2 — Quantidade de produtores e consumidores

- **1 produtor e 3 consumidores:** o buffer tende a ficar vazio, pois há mais consumidores disputando os itens.
- **3 produtores e 1 consumidor:** o buffer tende a ficar cheio, pois a produção pode superar o consumo.
- **3 produtores e 3 consumidores:** as velocidades tendem a ficar mais equilibradas, embora o escalonamento possa causar esperas.

A relação principal é: produção mais rápida enche o buffer; consumo mais rápido esvazia o buffer.

## Experimento 3 — `notify()` versus `notify_all()`

`notify()` acorda uma única thread, enquanto `notify_all()` acorda todas as threads que aguardam na condição. A thread acordada não continua imediatamente: primeiro precisa readquirir o lock. Depois disso, ela deve testar a condição no `while`, porque outra thread pode ter alterado o estado.

## Análise geral

O buffer limitado demonstra uma coordenação baseada em estado. As threads não esperam um tempo fixo; esperam até que exista espaço ou item disponível.

# Parte 4 — Outro caso de Condition: fila de pedidos

## Respostas e análise

O atendente espera quando a fila está vazia. O sistema adiciona um pedido dentro do bloco protegido por `Condition` e chama `notify()`. O atendente acordado retira o primeiro pedido com `pop(0)`, preservando a ordem de chegada.

A sentinela `None` representa o fim da produção de pedidos. Quando o atendente recebe essa sentinela, encerra sua execução.

## Trecho importante

```python
with condition_pedidos:
    while not pedidos:
        condition_pedidos.wait()
    pedido = pedidos.pop(0)
```

A espera ocorre sem consulta contínua e o lock fica disponível para a thread que recebe novos pedidos.

In [ ]:
import threading
import time

pedidos = []
condition_pedidos = threading.Condition()


def atendente():
    while True:
        with condition_pedidos:
            while not pedidos:
                print("Atendente: aguardando pedido...")
                condition_pedidos.wait()
            pedido = pedidos.pop(0)
        if pedido is None:
            print("Atendente: encerrando.")
            return
        print(f"Atendente: processando {pedido}")
        time.sleep(0.2)


def receber_pedidos():
    for numero in range(1, 4):
        time.sleep(0.3)
        with condition_pedidos:
            pedido = f"PED-{numero:03d}"
            pedidos.append(pedido)
            print(f"Sistema: {pedido} recebido.")
            condition_pedidos.notify()
    with condition_pedidos:
        pedidos.append(None)
        condition_pedidos.notify()


atendente_thread = threading.Thread(target=atendente)
recebedor_thread = threading.Thread(target=receber_pedidos)
atendente_thread.start()
recebedor_thread.start()
atendente_thread.join()
recebedor_thread.join()

# A fila e processada em ordem e None encerra o atendente.

# Parte 5 — Barrier

## Respostas

1. A ordem de chegada à barreira varia conforme o escalonamento das threads e os tempos de espera.
2. A primeira thread não inicia a Fase 2; ela fica bloqueada em `barrier.wait()`.
3. Quando a última thread chega, a barreira é liberada e todas podem avançar.

## Trecho importante

```python
barrier = threading.Barrier(NUM_THREADS)
barrier.wait()
```

## Análise

`Barrier` responde à pergunta **quando todos podem avançar?**. Ela não define a ordem da Fase 2 e não acelera o programa. Sua função é impedir que uma thread avance antes das demais.

In [ ]:
import random
import threading
import time

NUM_THREADS = 4
barrier = threading.Barrier(NUM_THREADS)


def tarefa(numero):
    print(f"Thread {numero}: Fase 1")
    time.sleep(random.uniform(0.1, 0.5))
    print(f"Thread {numero}: chegou à barreira")
    barrier.wait()
    print(f"Thread {numero}: Fase 2")


threads = [threading.Thread(target=tarefa, args=(numero,)) for numero in range(NUM_THREADS)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print("Todas as threads concluíram as duas fases.")

# Parte 6 — Barrier no carregamento de sensores

## Respostas

1. O arquivo que tiver menor carga ou menor atraso tende a terminar primeiro, mas a ordem pode variar.
2. Não. A thread que termina primeiro aguarda na barreira.
3. Todos os arquivos precisam ser carregados e todas as threads precisam chegar à barreira.

## Trecho importante

```python
fim_carga = time.perf_counter()
barrier_sensores.wait()
inicio_processamento = time.perf_counter()
```

## Análise

A barreira divide o trabalho em duas fases: carregamento e processamento. Mesmo que uma thread termine cedo, ela não processa seu arquivo antes da chegada das demais. A comparação entre `fim_carga` e `inicio_processamento` permite verificar essa garantia.

In [ ]:
import random
import threading
import time

arquivos = [(f"sensor_{numero:02d}.csv", quantidade) for numero, quantidade in enumerate([100, 250, 500, 1000, 750], 1)]
barrier_sensores = threading.Barrier(len(arquivos))
registros = {}
lock = threading.Lock()
inicio_execucao = time.perf_counter()


def carregar_e_processar(nome, quantidade):
    inicio_carga = time.perf_counter() - inicio_execucao
    print(f"[{inicio_carga:.2f}s] Iniciando carga: {nome}")
    time.sleep(random.uniform(0.1, 0.5))
    fim_carga = time.perf_counter() - inicio_execucao
    with lock:
        registros[nome] = {"inicio_carga": inicio_carga, "fim_carga": fim_carga}
    print(f"[{fim_carga:.2f}s] Carga concluida: {nome} ({quantidade} registros)")
    print(f"[{fim_carga:.2f}s] {nome} aguardando na barreira")
    barrier_sensores.wait()
    inicio_processamento = time.perf_counter() - inicio_execucao
    with lock:
        registros[nome]["inicio_processamento"] = inicio_processamento
    print(f"[{inicio_processamento:.2f}s] PROCESSANDO {nome}")


threads = [threading.Thread(target=carregar_e_processar, args=arquivo) for arquivo in arquivos]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

ultimo_carregamento = max(registro["fim_carga"] for registro in registros.values())
primeiro_processamento = min(registro["inicio_processamento"] for registro in registros.values())
print(f"Ultimo carregamento: {ultimo_carregamento:.2f}s")
print(f"Primeiro processamento: {primeiro_processamento:.2f}s")
assert primeiro_processamento >= ultimo_carregamento

# A afirmacao confirma que a Barrier impediu o processamento antecipado.

# Parte 7 — Experimentos dirigidos com Barrier

## Experimento 1 — Remover a Barrier

Sem `barrier.wait()`, uma thread pode começar a processar enquanto outra ainda carrega seu arquivo. A barreira serve exatamente para separar as fases. As médias numéricas não mudam, mas a coordenação e a ordem temporal mudam.

## Experimento 2 — Aumentar para cinco arquivos

Com cinco arquivos, a barreira precisa receber cinco participantes. Se somente quatro terminarem, nenhuma thread pode avançar para o processamento.

## Experimento 3 — Thread lenta

Se o sensor 4 dormir por 5 segundos e os demais dormirem por 1 segundo, o sensor 4 será o gargalo. A barreira não acelera a execução; ela faz as threads rápidas esperarem a mais lenta para manter a sincronização.

## Experimento 4 — Participante ausente

Uma barreira configurada com 4 participantes não é liberada por apenas 3 threads. Com `timeout`, as threads recebem `BrokenBarrierError`, evitando um bloqueio indefinido.

```python
try:
    barrier.wait(timeout=3)
except threading.BrokenBarrierError:
    print("Barreira quebrada")
```

# Parte 8 — Comparação dos mecanismos

| Mecanismo | Pergunta respondida | Aplicação |
|---|---|---|
| `Lock` | Quem pode entrar na região crítica? | Uma thread altera o estoque por vez. |
| `Semaphore(N)` | Quantas threads podem entrar? | Até `N` clientes usam um serviço simultaneamente. |
| `Condition` | Quando uma thread pode continuar? | Consumidor espera até existir um item. |
| `Barrier(N)` | Quando todos podem avançar? | Workers terminam a carga antes do processamento. |

## Respostas

1. Uma thread por vez alterando o estoque: `Lock`.
2. No máximo cinco clientes simultâneos: `Semaphore(5)`.
3. Consumidor aguardando um item: `Condition`.
4. Quatro workers terminando a carga antes da próxima fase: `Barrier(4)`.

## Análise

`Lock` protege uma região crítica. `Semaphore` limita capacidade simultânea. `Condition` coordena threads a partir de uma mudança de estado. `Barrier` sincroniza participantes entre fases. Eles podem ser combinados, mas resolvem problemas diferentes.

# Parte 9 — Exercícios finais

## Exercício 1 — Condition com contadores

### Configuração

- 3 produtores
- 2 consumidores
- `MAX_SIZE = 4`
- `NUM_ITEMS = 30`

### Resposta e análise

Os produtores incrementam `esperas_produtores` sempre que encontram o buffer cheio. Os consumidores incrementam `esperas_consumidores` sempre que encontram o buffer vazio.

Não existe uma quantidade fixa de esperas: ela depende do escalonamento das threads e das velocidades de produção e consumo. Se a produção for mais rápida, produtores tendem a esperar pelo consumo. Se o consumo for mais rápido, consumidores tendem a esperar pela produção. O tamanho 4 limita quanto trabalho pode ficar acumulado.

A conclusão deve ser baseada nos contadores obtidos na execução, e não em uma ordem predeterminada.

## Exercício 2 — Barrier com arquivos

Cada thread deve registrar quatro instantes:

1. início da carga;
2. término da carga;
3. chegada à barreira;
4. início do processamento.

### Resposta

Nenhuma thread inicia o processamento antes de o último arquivo terminar a carga. A `Barrier` só é liberada quando as cinco threads chegam a ela. Assim, o primeiro instante de processamento deve ser maior ou igual ao maior instante de término de carga.

## Síntese final

- `Lock` = exclusão mútua.
- `Semaphore` = limite de participantes simultâneos.
- `Condition` = espera por mudança de estado.
- `Barrier` = sincronização entre fases.

A escolha do mecanismo depende da pergunta que o programa precisa responder: proteger dados, limitar acesso, aguardar uma condição ou esperar todos os participantes.